# 01. Leíró Statisztika és Feltáró Adatelemzés (EDA)

**Cél:** az eladó lakáspiac fundamentális ár-, méret- és elrendezési struktúrájának statisztikai vizsgálata.

---

### 📖 A nyers átlagárak torzító hatása
A lakáspiaci elemzések gyakori hibája a számtani átlag kizárólagos használata. A piac extrém módon heterogén: a standard garzonoktól az egyedi árazású luxusingatlanokig terjed, az eloszlás felső szegmense hosszú „farkat" képez, amely az átlagot felfelé torzítja. A fejezet az árak valódi eloszlását, a robusztus kvartiliseket és a logaritmikus transzformáció indokoltságát mutatja be.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from scipy import stats

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Eladó lakások száma az EDA mintában: {len(elado)} db.")

### 1. Középértékek és Szóródási Mutatók

**📌 Az eredmények értelmezése:**
- **Átlag vs. medián:** a kettő közötti eltérés pontosan számszerűsíti a felső árszegmens torzító hatását; a piac valódi árszintjét a medián négyzetméterár reprezentálja.
- **Relatív szórás:** a magas (20% feletti) relatív szórás arra utal, hogy az árakat dominánsan a lokáció és az épületminőség diverzitása magyarázza, nem csupán az alapterület.

**🔍 A kvartilisek jelentősége:** Q1 a „belépő" árszint, Q3 a prémium szegmens határa, az IQR (interkvartilis terjedelem) a piac középső 50%-ának szélsőérték-mentes sávszélessége.

In [ ]:
kpi_cards = [
    ("Átlagos Kínálati Ár", fmt_mft(elado['price_huf'].mean() / 1e6), f"Medián: {fmt_mft(elado['price_huf'].median() / 1e6)}", "#1e3a8a"),
    ("Átlagos Ár / m²", fmt_huf(elado['nm_ar_huf'].mean()), f"Medián: {fmt_huf(elado['nm_ar_huf'].median())}", "#2563eb"),
    ("Átlagos Alapterület", f"{elado['alapterulet_nm'].mean():.1f} m²", f"Medián: {elado['alapterulet_nm'].median():.1f} m²", "#059669"),
    ("Átlagos Szobaszám", f"{elado['szobaszam_osszes'].mean():.1f} szoba", f"Módusz: {elado['szobaszam_osszes'].mode()[0]} szoba", "#d97706"),
    ("Ár / m² Szórás", fmt_huf(elado['nm_ar_huf'].std()), f"Relatív szórás: {elado['nm_ar_huf'].std()/elado['nm_ar_huf'].mean()*100:.1f}%", "#dc2626"),
    ("Ár / m² Ferdeség", f"{stats.skew(elado['nm_ar_huf'].dropna()):.2f}", "Jobbra ferde (log-normális)", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))

# Részletes leíró táblázat
desc_vars = ['price_huf', 'nm_ar_huf', 'alapterulet_nm', 'szobaszam_osszes']
desc_names = ['Ár (HUF)', 'Ár / m² (HUF)', 'Alapterület (m²)', 'Szobaszám']
stat_df = elado[desc_vars].describe().T
stat_df['skew'] = [stats.skew(elado[c].dropna()) for c in desc_vars]
stat_df['kurtosis'] = [stats.kurtosis(elado[c].dropna()) for c in desc_vars]
stat_df.index = desc_names
stat_df.columns = ['Darab', 'Átlag', 'Szórás', 'Min', '25% (Q1)', 'Medián (Q2)', '75% (Q3)', 'Max', 'Ferdeség', 'Csúcsosság']

html_table = "<div style='overflow-x:auto; margin: 15px 0;'>" + stat_df.round(2).to_html(classes='table table-bordered table-hover') + "</div>"
display(HTML(html_table))

### 2. Áreloszlás és Logaritmikus Transzformáció

**📌 A hisztogramok összehasonlítása:**
- **Bal ábra:** a nyers fajlagos árak eloszlása — tipikusan jobbra ferde.
- **Jobb ábra:** a természetes logaritmus (ln(ár/m²)) alkalmazásával kapott eloszlás — közelíti a normálist.

**🔍 Ökonometriai indoklás:**
1. A lineáris regresszió (OLS) alapfeltétele a hibatagok normál eloszlása; a nyers árak aszimmetriája sérti ezt, a log-transzformáció „kisimítja".
2. A logaritmikus függő változó mellett a becsült együtthatók közvetlenül **százalékos prémiumként/diszkontként** (marginal willingness to pay) értelmezhetők.

In [ ]:
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Nyers Négyzetméterár Eloszlása (Ft/m²)', 'Logaritmikus Négyzetméterár Eloszlása (ln(Ár/m²))')
)

fig1.add_trace(
    go.Histogram(x=elado['nm_ar_huf'], nbinsx=35, name='Ár/m²', marker_color='#2563eb', opacity=0.8),
    row=1, col=1
)

fig1.add_trace(
    go.Histogram(x=np.log(elado['nm_ar_huf']), nbinsx=35, name='ln(Ár/m²)', marker_color='#059669', opacity=0.8),
    row=1, col=2
)

fig1.update_layout(
    title_text='Fajlagos árak eloszlása és normálissá alakítása log-transzformációval',
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False
)
fig1.show()

### 3. Dobozdiagramok (Boxplot) és Kiugró Értékek Városrészenként

**📌 Hogyan kell olvasni a dobozdiagramot?**
- **A doboz közepe (vastag vonal)**: A medián ár (a középső lakás értéke).
- **A doboz alsó és felső éle**: Az alsó kvartilis (Q1) és felső kvartilis (Q3). A doboz magassága maga a piac középső 50%-a (IQR).
- **A függőleges bajszok**: A normális ártartomány határai ($1.5 \times \text{IQR}$).
- **A bajszokon kívüli egyedi pontok**: Statisztikai kiugró értékek (outlierek).

**🔍 Mit látunk Kőbánya városrészeiben?**
- **Óhegy és Ligettelek**: A dobozok magasabban helyezkednek el, a felső kvartilis eléri az 1.3 - 1.4 millió Ft/m²-es szintet.
- **Újhegy**: A doboz rendkívül tömör és feszes! A lakások szabványos méretei és paneles jellege miatt az árak alacsony szórásúak, a medián ~1.1 millió Ft/m² körül stabil.
- **Gyárdűlő és Laposdűlő**: Hosszabb dobozok és nagyobb szóródás, mivel a vegyes ipari-rozsdaövezeti környezetben az elhanyagolt épületek és a modern új fejlesztések árai erősen eltérnek egymástól.

**💡 Tanulság**: Újhegy a legkiszámíthatóbb árú övezet, Óhegy a legváltozatosabb prémium zóna.

In [ ]:
# Városrészi boxplot
fig2 = px.box(
    elado,
    x='varosresz',
    y='nm_ar_huf',
    color='varosresz',
    title='Négyzetméterár dobozábra (Boxplot) városrészenként',
    labels={'varosresz': 'Városrész', 'nm_ar_huf': 'Ár / m² (HUF)'},
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(xaxis_tickangle=-30, height=450, showlegend=False)
fig2.show()

# Méret vs Ár szórásdiagram OLS trenddel
fig3 = px.scatter(
    elado,
    x='alapterulet_nm',
    y='ar_millio_ft',
    color='varosresz',
    trendline='ols',
    title='Alapterület vs. Kínálati Ár összefüggés városrészenként',
    labels={'alapterulet_nm': 'Alapterület (m²)', 'ar_millio_ft': 'Ár (Millió Ft)', 'varosresz': 'Városrész'},
    template=PLOTLY_TEMPLATE
)
fig3.update_layout(height=480)
fig3.show()

### 4. Korrelációs Hőtérkép és Változókapcsolatok

**📌 Mit jelent a korrelációs mátrix?**
A változók közötti lineáris kapcsolatot méri -1 és +1 között:
- **Pozitív érték (kék)**: Együtt mozognak (pl. alapterület és teljes ár: $r \approx +0.85$).
- **Negatív érték (piros)**: Ellentétesen mozognak (pl. alapterület és m² ár: a garzon-effektus miatt a nagyobb lakások m² ára enyhén alacsonyabb).
- **Panelszerkezet (is_panel)**: Szignifikáns negatív kapcsolatot mutat a négyzetméterárral.

**💡 Fő tanulság**: A méret, a panel szerkezet, az állapot és a közlekedési csomópontok távolsága a kőbányai piac legfőbb mozgatórugói.

In [ ]:
# Interaktív Többváltozós Eloszláselemzés (Plotly updatemenus)
eda_vars = [
    ('nm_ar_huf', '1. Fajlagos Négyzetméterár (HUF/m²)', 40),
    ('ar_millio_ft', '2. Kínálati Vételár (M Ft)', 30),
    ('alapterulet_nm', '3. Lakás Alapterület (m²)', 25),
    ('szobaszam_osszes', '4. Szobaszám (db)', 15)
]

fig_eda = go.Figure()
buttons = []

for i, (col, label, nbins) in enumerate(eda_vars):
    sub_fig = px.histogram(
        elado, x=col, color='varosresz', barmode='overlay', opacity=0.7,
        nbins=nbins, template=PLOTLY_TEMPLATE
    )
    for tr in sub_fig.data:
        tr.visible = (i == 0)
        fig_eda.add_trace(tr)

traces_per_var = len(elado['varosresz'].unique())

for i, (col, label, nbins) in enumerate(eda_vars):
    vis = [False] * len(fig_eda.data)
    for t_idx in range(i * traces_per_var, (i + 1) * traces_per_var):
        if t_idx < len(vis): vis[t_idx] = True
    buttons.append(dict(
        label=label,
        method='update',
        args=[{'visible': vis}, {'title': f'{label} eloszlása a kőbányai városrészekben', 'xaxis': {'title': label}}]
    ))

fig_eda.update_layout(
    title=f'{eda_vars[0][1]} eloszlása a kőbányai városrészekben',
    xaxis_title=eda_vars[0][1],
    yaxis_title='Gyakoriság (darabszám)',
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        direction='down',
        x=0.01, y=0.99, xanchor='left', yanchor='top',
        bgcolor='white', bordercolor='#cbd5e1'
    )],
    template=PLOTLY_TEMPLATE,
    height=480
)
fig_eda.show()